In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')
print(sys.path)

['/home/thule/miniconda3/envs/open-clip-training/lib/python311.zip', '/home/thule/miniconda3/envs/open-clip-training/lib/python3.11', '/home/thule/miniconda3/envs/open-clip-training/lib/python3.11/lib-dynload', '', '/home/thule/miniconda3/envs/open-clip-training/lib/python3.11/site-packages', '../']


In [2]:
import torch
import tqdm
from sklearn.metrics import classification_report
import numpy as np
from sklearn.metrics import confusion_matrix

In [3]:
seed = 1
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

### window_size=32

In [5]:
from dataset import CANMLDataset

# window_size = 32
test_dataset = CANMLDataset(
        root_dir="../data/can-ml/can-ml/2017-subaru-forester/020926/preprocessed-keep-per-frame-label/rid1/TFRecord_w32_s32/", window_size=32, stride_size=32, is_train=False, is_test=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=True, pin_memory=True)

In [12]:
import os
import numpy as np
from sklearn.model_selection import train_test_split

# ---------------- Configuration ----------------
# window_size=32
# DATA_DIR = "../data/can-ml/can-ml/2017-subaru-forester/020926/preprocessed-keep-per-frame-label/rid1/TFRecord_w32_s32/" 
# window_size=16 
# DATA_DIR = "../data/can-ml/can-ml/2017-subaru-forester/100126/preprocessed/rid1/TFRecord_w16_s16/"
# window_size=64
DATA_DIR = "../data/can-ml/can-ml/2017-subaru-forester/150926/preprocessed/rid1/TFRecord_w64_s64/"
OUT_DIR = DATA_DIR + "val-test-split/"             # where to save the new files
SEED = 42

# ---------------- Load ----------------
x = np.load(os.path.join(DATA_DIR, "xtest-64-64.npy"))
y = np.load(os.path.join(DATA_DIR, "ytest-64-64.npy"))
assert len(x) == len(y), f"Mismatched lengths: x={len(x)}, y={len(y)}"
print(f"Loaded x: {x.shape}, y: {y.shape}")

# ---------------- Stratified 50/50 split ----------------
# stratify=y keeps the proportion of every class identical in both halves,
# so each set contains all label values (normal and every attack type).
classes, counts = np.unique(y, return_counts=True)
if counts.min() < 2:
    raise ValueError(
        f"Classes {classes[counts < 2]} have fewer than 2 samples and cannot "
        "appear in both sets."
    )

x_val, x_test_new, y_val, y_test_new = train_test_split(
    x, y, test_size=0.5, stratify=y, random_state=SEED, shuffle=True
)

# ---------------- Check the class distribution ----------------
print(f"\n{'Class':>6} | {'Original':>9} | {'Val':>8} | {'Test':>8}")
print("-" * 42)
for c, n in zip(classes, counts):
    print(f"{c:>6} | {n:>9} | {(y_val == c).sum():>8} | {(y_test_new == c).sum():>8}")
print("-" * 42)
print(f"{'Total':>6} | {len(y):>9} | {len(y_val):>8} | {len(y_test_new):>8}")

# ---------------- Save ----------------
np.save(os.path.join(OUT_DIR, "xval-64-64.npy"), x_val)
np.save(os.path.join(OUT_DIR, "yval-64-64.npy"), y_val)
np.save(os.path.join(OUT_DIR, "xtest-64-64.npy"), x_test_new)
np.save(os.path.join(OUT_DIR, "ytest-64-64.npy"), y_test_new)
print(f"\nSaved x_val.npy, y_val.npy, x_test_split.npy, y_test_split.npy to {OUT_DIR}")

Loaded x: (17476, 64, 9), y: (17476,)

 Class |  Original |      Val |     Test
------------------------------------------
   0.0 |     13966 |     6983 |     6983
   1.0 |       892 |      446 |      446
   2.0 |       134 |       67 |       67
   3.0 |       170 |       85 |       85
   4.0 |       141 |       71 |       70
   5.0 |      1273 |      637 |      636
   6.0 |       266 |      133 |      133
   7.0 |       319 |      159 |      160
   8.0 |       124 |       62 |       62
   9.0 |       191 |       95 |       96
------------------------------------------
 Total |     17476 |     8738 |     8738

Saved x_val.npy, y_val.npy, x_test_split.npy, y_test_split.npy to ../data/can-ml/can-ml/2017-subaru-forester/150926/preprocessed/rid1/TFRecord_w64_s64/val-test-split/
